# Quality of Life and Happiness Analysis

### Abstract

This project investigates how external socioeconomic factors are associated with Quality of Life and Happiness Score across different countries over the period from 2015 to 2022. Two datasets were merged — the World Happiness Report and the Quality of Life Index by Country — to enable a comprehensive cross-analysis of indicators such as Social Support, Freedom, Generosity, Trust in Government, Purchasing Power, Safety, and Pollution. The analysis includes descriptive statistics, trend visualization, correlation analysis, and hypothesis testing aimed at identifying which factors are most strongly associated with well-being at a national level.

The project was completed by a team of two. Visualizations and descriptive analysis of the World Happiness dataset were handled primarily by Sofia, while analysis of the Quality of Life dataset was handled primarily by Anastasia. The merged dataset analysis, cleanup, hypothesis formulation and hypothesis testing were completed collaboratively by both team members.

# 1. World Happiness Dataset

### Dataset Description


Dataset source:

https://www.kaggle.com/datasets/hari31416/world-happiness-report/data

The datasets belong to the field of social and economic statistics.

They contain annual data for multiple countries from 2015 to 2022 and include indicators such as economic conditions (GDP per capita), social support (Family), healthy life expectancy (Health), freedom to make life choices (Freedom), generosity, trust in institutions, and overall happiness scores.

For further analysis, the datasets will be merged into a single table using the additional Year column, which allows matching records from different years and enables time-series analysis.

In [ ]:
import pandas as pd

In [ ]:
tables = []
years = range(2015, 2023)

for year in years:
    table = pd.read_csv(f"../data_sets/{year}.csv")
    table["Year"] = year
    tables.append(table)


The annual World Happiness Report tables for 2015–2022 are loaded and combined into a list. An additional Year column is added to each table to preserve the original observation year and enable time-based analysis after merging.

In [ ]:
reference_year = tables[0]

for i in range(len(tables)):
    if tables[i].columns.equals(reference_year.columns):
        print(f"{years[i]}: equal")
    else:
        print(f"{years[i]}: {set(tables[i].columns)^set(reference_year.columns)}")

The comparison of column names showed that the datasets do not have completely identical structures: the datasets for 2018 and 2019 do not contain the Dystopia column, while all other years include it.

Since the purpose of this project is to analyze variables that are available for every year, the datasets will be combined using only the columns that are present in all tables.

In [ ]:
all_data = pd.concat(tables, axis = 0, join = 'inner', ignore_index = True)

In [ ]:
all_data.head()

In [ ]:
all_data.shape

In [ ]:
all_data.info()

The resulting dataset contains 1185 rows and 11 fields. The fields are represented by the following data types:
- 2 categorical fields (object): Country, Region
- 2 integer numerical fields (int64): Happiness Rank, Year
- 7 continuous numerical fields (float64): Happiness Score, Economy, Family, Health, Freedom, Generosity, Trust

In [ ]:
all_data.isna().sum()

In [ ]:
all_data.duplicated().sum()

The dataset is generally of high quality.

* No duplicate rows were found in the dataset.
* Only one missing value was detected in the Trust column, while all other fields were complete.
* The data types of all columns were examined and found to be appropriate for their contents: categorical variables are stored as object, numerical indicators as float64, and ranking and year values as int64. No inconsistent values or data type issues were
 identified.

 Therefore, the dataset can be considered suitable for further analysis.


### Descriptive statistics

In [ ]:
statistics_h = pd.DataFrame ({

    "Mean":[
    all_data["Happiness Score"].mean(),
    all_data["Economy"].mean(),
    all_data["Family"].mean(),
    all_data["Health"].mean(),
    all_data["Freedom"].mean(),
    all_data["Generosity"].mean(),
    all_data["Trust"].mean()],

    "Median": [
    all_data["Happiness Score"].median(),
    all_data["Economy"].median(),
    all_data["Family"].median(),
    all_data["Health"].median(),
    all_data["Freedom"].median(),
    all_data["Generosity"].median(),
    all_data["Trust"].median()],

    "Standard Deviation": [
    all_data["Happiness Score"].std(),
    all_data["Economy"].std(),
    all_data["Family"].std(),
    all_data["Health"].std(),
    all_data["Freedom"].std(),
    all_data["Generosity"].std(),
    all_data["Trust"].std()],

    "Minimum": [
    all_data["Happiness Score"].min(),
    all_data["Economy"].min(),
    all_data["Family"].min(),
    all_data["Health"].min(),
    all_data["Freedom"].min(),
    all_data["Generosity"].min(),
    all_data["Trust"].min()],

    "Maximum": [
    all_data["Happiness Score"].max(),
    all_data["Economy"].max(),
    all_data["Family"].max(),
    all_data["Health"].max(),
    all_data["Freedom"].max(),
    all_data["Generosity"].max(),
    all_data["Trust"].max()],
},

index = ["The happiness score on a scale from 0 to 10",
         "The GDP per capita index", "Social Support",
         "Healthy Life Expectancy",
         "Freedom to make life choices",
         "Generosity in the community",
         "Trust in government"]
)

statistics_h

The descriptive statistics provide a general overview of the dataset and the distribution of its main numerical variables. Most indicators have reasonable ranges and average values that correspond to the expected scale of the World Happiness Report data.

However, the summary statistics also revealed potential inconsistencies in some variables. In particular, certain indicators show unusually large differences between their average values, medians, and maximum values, which may indicate that different measurement scales were used across parts of the dataset. These discrepancies will be investigated in more detail during the Data Cleanup stage, where the data will be checked for inconsistencies and, if necessary, standardized before further analysis.


### Data cleanup

During the initial exploration of the dataset, several data quality issues were identified:

* First, one missing value was found in the Trust column, while all other fields were complete.
* Second, the descriptive statistics revealed unusually large standard deviations and extreme values in the Health and Economy indicators.

These results suggest the presence of inconsistencies in the data, possibly caused by differences in measurement scales across years.

To ensure the reliability of further analysis, these issues will be investigated and addressed in the Data Cleanup stage. Missing values will be handled appropriately, and the identified inconsistencies will be examined and corrected where necessary.


In [ ]:
all_data = all_data.dropna()
all_data.isna().sum()
all_data

In [ ]:
all_data.groupby("Year")["Health"].describe()

In [ ]:
all_data.groupby("Year")["Economy"].describe()

In [ ]:
all_data = all_data.drop(columns=["Health", "Economy"])
all_data.columns

In [ ]:
all_data.groupby("Country")["Year"].nunique().value_counts().sort_index()

Some countries have data available for only a limited number of years. To improve the reliability and comparability of the analysis, only countries with observations for 8 years were retained. This approach reduces the impact of countries with insufficient data while preserving a substantial portion of the dataset.

In [ ]:
years_countries = all_data.groupby("Country")["Year"].nunique()
years_countries.value_counts().sort_index()

In [ ]:
good_countries = years_countries[years_countries == 8].index
all_data = all_data[all_data["Country"].isin(good_countries)]
all_data.head()

Some countries have data for only a few years. To ensure more reliable analysis, only countries with observations available for all years in the dataset were retained.

In [ ]:
print(all_data.shape)
display(all_data.dtypes)

After cleaning and filtering, the dataset contains 928 rows and 9 columns. All remaining columns have appropriate data types for further analysis.

### Plots

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(9,7))

sns.histplot(
    data=all_data,
    x="Generosity",
    bins=20,
    kde=True,
    color = "#FF69B4"
)

plt.title("Distribution of Generosity")
plt.xlabel("Generosity")
plt.ylabel("Count")

plt.show()

The histogram shows the distribution of the Generosity indicator across all countries and years included in the dataset.
 The distribution is right-skewed, with most values concentrated between -0.2 and 0.4, and a long tail extending toward higher values. This suggests that while most countries report relatively low or moderate levels of generosity, a smaller number of countries stand out with noticeably higher scores.

In [ ]:
plt.figure(figsize=(9,7))
plt.scatter(all_data["Freedom"], all_data["Happiness Score"],
            alpha=0.5, color="#98FB98")
plt.title("Freedom vs Happiness Score")
plt.xlabel("Freedom")
plt.ylabel("Happiness Score")
plt.grid(alpha=0.3)
plt.show()

The scatter plot reveals a positive relationship between freedom and happiness — countries where people report higher freedom to make life choices tend to score higher on the happiness scale.

In [ ]:
plt.figure(figsize=(9,7))

sns.histplot(
    data=all_data,
    x="Family",
    bins=20,
    kde=True,
    color="#A8DADC"
)

plt.title("Distribution of Social Support")
plt.xlabel("Social Support")
plt.ylabel("Count")

plt.show()

The histogram shows the distribution of the Social Support indicator across all countries and years in the dataset. Most observations are concentrated between 0.8 and 1.3, indicating that moderate to high levels of social support are common among countries. Very low values occur relatively rarely, while the distribution suggests that social support is generally well developed in most observations included in the dataset.


In [ ]:
mean_happiness = (
    all_data.groupby("Year")["Happiness Score"]
    .mean()
    .reset_index()
)
plt.figure(figsize=(8,5))

sns.lineplot(
    data=mean_happiness,
    x="Year",
    y="Happiness Score",
    color = "pink",
    marker="o",
    linewidth=3
)

plt.title("Average Happiness Score by Year")
plt.xlabel("Year")
plt.ylabel("Average Happiness Score")

plt.show()

The line plot displays the trend in average Happiness Score from 2015 to 2022. The average score shows a gradual increase from 2015 to 2021, rising from approximately 5.5 to 5.7. A slight decline is observed in 2022, although the overall trend remains positive.

# 2. Quality of Life Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv("../data_sets/quality_of_life_indices_by_country.csv")

### Dataset description

Dataset source:

https://www.kaggle.com/datasets/marcelobatalhah/quality-of-life-index-by-country/data

The dataset belongs to the socio-economic domain and contains quality of life indicators for different countries between 2015 and 2024. The dataset describes different aspects of living conditions and well-being across countries.

In [ ]:
data.head()

In [ ]:
data.shape

The dataset contains 1495 rows and 12 columns.

There is one categorical field (Country), one ranking field (Rank), one time-related field (Year), and nine numerical indicators describing different aspects of quality of life.

These indicators include the overall Quality of Life Index, Purchasing Power Index, Safety Index, Health Care Index, Cost of Living Index, Property Price to Income Ratio, Traffic Commute Time Index, Pollution Index, and Climate Index.

In [ ]:
data.info()

According to the data types, most variables are numerical. However, the Year and Climate Index columns are stored as strings, indicating potential formatting issues that require further inspection.

In [ ]:
data.isna().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
print("Non-numeric Climate Index values:", (data["Climate Index"] == "-").sum())

In [ ]:
data["Year"].unique()

The dataset is generally of good quality. No explicit missing values or duplicate rows were detected.

However, some inconsistencies were found. The Year column contains values such as "2015/2", while the Climate Index column contains 143 non-numeric entries represented by "-". Because of these formatting inconsistencies, both columns are interpreted as strings rather than purely numerical variables.

### Descriptive statistics

In [ ]:
statistics = pd.DataFrame({
    "Mean":[data["Quality of Life Index"].mean(),
            data["Purchasing Power Index"].mean(),
            data["Safety Index"].mean(),
            data["Health Care Index"].mean(),
            data["Cost of Living Index"].mean()],

    "Median":[
        data["Quality of Life Index"].median(),
        data["Purchasing Power Index"].median(),
        data["Safety Index"].median(),
        data["Health Care Index"].median(),
        data["Cost of Living Index"].median()],

    "Standard Deviation":
        [
        data["Quality of Life Index"].std(),
        data["Purchasing Power Index"].std(),
        data["Safety Index"].std(),
        data["Health Care Index"].std(),
        data["Cost of Living Index"].std()
        ],

    "Minimum":
        [
        data["Quality of Life Index"].min(),
        data["Purchasing Power Index"].min(),
        data["Safety Index"].min(),
        data["Health Care Index"].min(),
        data["Cost of Living Index"].min()
        ],
    "Maximum":
        [
        data["Quality of Life Index"].max(),
        data["Purchasing Power Index"].max(),
        data["Safety Index"].max(),
        data["Health Care Index"].max(),
        data["Cost of Living Index"].max()
        ]
},
    index=["Quality of Life Index",
           "Purchasing Power Index",
           "Safety Index",
           "Health Care Index",
           "Cost of Living Index"]
)

statistics.round(2)

Among the selected indicators, the Quality of Life Index has the highest variability, while the Health Care Index shows the lowest variability.

The Quality of Life Index also has the widest range of values, indicating substantial differences in living conditions across countries. The mean and median values are relatively close for several indicators, suggesting that most distributions are not strongly skewed.

### Data cleanup

In [ ]:
clean_data = data.drop(columns = "Climate Index")
clean_data.head()

Since the Climate Index column contains “-” for all observations from 2015, removing these values would result in losing an entire year of data. Therefore, the Climate Index column was removed from the dataset. Additionally, this indicator was not intended to be used in the correlation analysis or visualizations performed later in the project.

In [ ]:
clean_data["Year"] = clean_data["Year"].str.replace("/2", "")
clean_data["Year"] = clean_data["Year"].astype(int)

display(clean_data["Year"].unique())
display(clean_data.dtypes)

In [ ]:
clean_data = clean_data.groupby(["Country","Year"], as_index=False).mean()
clean_data.head(15)

The dataset contains semi-annual observations marked with the “/2” suffix. Since this project focuses on yearly analysis, the suffix was removed from the Year column and the values were converted to integers. Then, observations for the same country and year were combined by calculating the mean of the numerical indicators.

In [ ]:
clean_data = clean_data.drop(columns=["Rank"])
clean_data.head()

After grouping the data by Country and Year, the Rank column contained averaged values such as 37.5 and 59.5. Since ranking positions are ordinal measures and averaging them does not produce meaningful results, the Rank column was removed from further analysis.

In [ ]:
years_countries2 = clean_data.groupby("Country")["Year"].nunique()
years_countries2.value_counts().sort_index()

In [ ]:
good_countries2 = years_countries2[years_countries2 == 10].index
clean_data = clean_data[clean_data["Country"].isin(good_countries2)]
clean_data.head()

Some countries have data for only a few years. To ensure more reliable analysis, only countries with observations available for all years in the dataset were retained.

In [ ]:
print(clean_data.shape)
display(clean_data.dtypes)

After cleaning and filtering, the dataset contains 610 rows and 10 columns. All remaining columns have appropriate data types for further analysis.

### Plots

In [ ]:
plt.figure(figsize=(9,7))
plt.hist(clean_data["Quality of Life Index"], bins = 20, edgecolor="black", color='#1D4E89')

plt.title("Distribution of Quality of Life Index")
plt.xlabel("Quality of Life Index")
plt.ylabel("Frequency")

plt.grid(axis="y", alpha = 0.3)

plt.show()


Most observations have Quality of Life Index values between 90 and 190 points, while extremely low and extremely high values are relatively rare.

In [ ]:
plt.figure(figsize=(9,7))

clean_data.boxplot(
    column="Quality of Life Index",
    by="Year"
)

plt.suptitle("")
plt.title("Quality of Life Index by Year")
plt.xlabel("Year")
plt.ylabel("Quality of Life Index")

plt.grid(alpha=0.3)

plt.show()

The median Quality of Life Index remains relatively stable between 2015 and 2024, showing only minor fluctuations over time. The spread of values across countries is also fairly consistent throughout the observed period, indicating a similar level of variability between countries in most years.

The year 2015 stands out with a noticeably wider range of values, suggesting greater differences in quality of life across countries. Several low-value outliers can also be observed in some years.

In [ ]:
top = (
    clean_data.groupby('Country')['Quality of Life Index'].mean().sort_values(ascending=False).head(10)
)

plt.figure(figsize=(9,7))

top.sort_values().plot(kind='barh', color='#1D4E89')

plt.title('Top 10 Countries by Quality of Life Index')
plt.xlabel('Quality of Life Index')
plt.ylabel('Country')

plt.show()

The chart presents the top 10 countries by Quality of Life Index. Switzerland, Denmark, and Finland achieve the highest scores, while most top-ranked countries are located in Europe.

In [ ]:
yearly_pp = clean_data.groupby('Year')['Purchasing Power Index'].mean()

plt.figure(figsize=(9,7))

plt.plot(
    yearly_pp.index,
    yearly_pp.values,
    marker='o',
    color='#1D4E89',
)

plt.title('Average Purchasing Power Index by Year')
plt.xlabel('Year')
plt.ylabel('Purchasing Power Index')

plt.grid(alpha=0.3)
plt.show()

The line plot shows the average Purchasing Power Index over time. The indicator increased slightly between 2015 and 2016, then declined until 2021. After reaching its lowest value in 2021, purchasing power began to recover and increased steadily through 2024.

In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(
    clean_data['Purchasing Power Index'],
    clean_data['Quality of Life Index'],
    alpha=0.6, color='#1D4E89'
)

plt.title('Purchasing Power Index vs Quality of Life Index')
plt.xlabel('Purchasing Power Index')
plt.ylabel('Quality of Life Index')

plt.grid(alpha=0.3)
plt.show()

The scatter plot reveals a strong positive correlation between Purchasing Power Index and Quality of Life Index. Higher purchasing power is generally associated with better quality of life.

# 3. Merged Dataset

In [ ]:
final_data = pd.merge(clean_data, all_data, on = ["Country", "Year"], how="inner")

### Dataset Description

The two datasets were merged using an inner join on the Country and Year columns, creating a unified dataset for subsequent analysis.

In [ ]:
final_data.head()

In [ ]:
final_data.shape

In [ ]:
final_data.info()

In [ ]:
final_data.isna().sum()

In [ ]:
final_data.duplicated().sum()

In [ ]:
final_data["Year"].unique()

In [ ]:
final_data.groupby("Country")["Year"].count().value_counts()

After merging the datasets, the final dataset contains 440 observations and 17 variables. No missing values or duplicate records were found. The dataset covers the period from 2015 to 2022 and includes 55 countries with complete information for all eight years. The resulting dataset is suitable for further statistical analysis and visualization.

### Data cleanup

In [ ]:
final_data = final_data.drop(columns=["Happiness Rank"])
final_data.head()

The Happiness Rank column was removed from the final dataset, as it was not required for the analyses performed in this project.

### Detailed overview

In [ ]:
region_happiness = final_data.groupby("Region")["Happiness Score"].median().sort_values()

plt.figure(figsize=(9,7))
region_happiness.plot(kind="barh", color="#FFB6C1")
plt.title("Median Happiness Score by Region")
plt.xlabel("Median Happiness Score")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

The chart shows significant regional variation in median Happiness Scores. Australia and New Zealand, North America, and Western Europe rank highest, whereas South Asia and Sub-Saharan Africa report the lowest median values.

In [ ]:
trend = final_data.groupby("Year")[["Freedom", "Generosity", "Trust"]].mean().reset_index()

plt.figure(figsize=(9,7))
plt.plot(trend["Year"], trend["Freedom"], marker="o", label="Freedom", color="#87CEFA")
plt.plot(trend["Year"], trend["Generosity"], marker="s", label="Generosity", color="#FF69B4")
plt.plot(trend["Year"], trend["Trust"], marker="^", label="Trust in Government", color="#98FB98")
plt.title("Average Freedom, Generosity and Trust by Year")
plt.xlabel("Year")
plt.ylabel("Average Value")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The graph illustrates changes in Freedom, Generosity and Trust over time. Freedom and Trust increase significantly in 2020–2021, whereas Generosity drops to its lowest level during the same period. By 2022, all indicators move closer to their earlier values.

In [ ]:
first_year = final_data["Year"].min()
last_year = final_data["Year"].max()

data_first = final_data[final_data["Year"] == first_year]
data_last = final_data[final_data["Year"] == last_year]

top_first = data_first.nlargest(10, "Happiness Score")[["Country", "Happiness Score"]]
top_last = data_last.nlargest(10, "Happiness Score")[["Country", "Happiness Score"]]

first_countries = set(top_first["Country"])
last_countries = set(top_last["Country"])

colors_first = ["green" if c in last_countries else "skyblue" for c in top_first["Country"]]
colors_last = ["green" if c in first_countries else "pink" for c in top_last["Country"]]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].barh(top_first["Country"], top_first["Happiness Score"], color=colors_first)
axes[0].invert_yaxis()
axes[0].set_title(f"Top-10 by Happiness ({first_year})\nGreen = also in top-10 in {last_year}")
axes[0].set_xlabel("Happiness Score")

axes[1].barh(top_last["Country"], top_last["Happiness Score"], color=colors_last)
axes[1].invert_yaxis()
axes[1].set_title(f"Top-10 by Happiness ({last_year})\nGreen = also in top-10 in {first_year}")
axes[1].set_xlabel("Happiness Score")

plt.suptitle(f"Comparison: Top-10 Happiest Countries, {first_year} vs {last_year}", fontsize=14)
plt.tight_layout()
plt.show()

The composition of the top 10 happiest countries remained highly stable between 2015 and 2022. Most countries appear in both rankings, indicating persistent differences in happiness levels across countries over time.

In [ ]:
trend_economy = (
    final_data.groupby("Year")[[
        "Purchasing Power Index",
        "Cost of Living Index",
        "Property Price to Income Ratio"
    ]]
    .mean()
    .reset_index()
)

plt.figure(figsize=(9,7))

plt.plot(trend_economy["Year"], trend_economy["Purchasing Power Index"], marker="o", label="Purchasing Power Index", color = "#87CEFA")
plt.plot(trend_economy["Year"], trend_economy["Cost of Living Index"], marker="s", label="Cost of Living Index", color = "#FF69B4")
plt.plot(trend_economy["Year"], trend_economy["Property Price to Income Ratio"], marker="^", label="Property Price to Income Ratio", color = "#FFB6C1")

plt.title("Average Economic Indices by Year")
plt.xlabel("Year")
plt.ylabel("Average Index Value")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The Purchasing Power Index shows an overall downward trend between 2015 and 2022. The Cost of Living Index remains relatively stable throughout the period, while the Property Price to Income Ratio gradually increases.

In [ ]:
leaders = (
    final_data.loc[final_data.groupby("Year")["Happiness Score"].idxmax()]
    [["Year", "Country", "Happiness Score"]]
    .reset_index(drop=True)
)

plt.figure(figsize=(9,7))
bars = plt.bar(leaders["Year"].astype(str), leaders["Happiness Score"], color="#FF69B4")

for bar, country in zip(bars, leaders["Country"]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             country, ha="center", va="bottom", rotation=45, fontsize=9)

plt.title("Happiness Leader by Year")
plt.xlabel("Year")
plt.ylabel("Happiness Score")
plt.ylim(0, leaders["Happiness Score"].max() + 1)
plt.tight_layout()
plt.show()

The chart presents the country with the highest Happiness Score each year. Switzerland, Denmark and Norway led the ranking in the early years, while Finland remained the happiest country from 2018 to 2022.

In [ ]:
median_happiness = final_data["Happiness Score"].median()

high_happiness = final_data[final_data["Happiness Score"] >= median_happiness]
low_happiness = final_data[final_data["Happiness Score"] < median_happiness]

comparison = pd.DataFrame({
    "High Happiness": [
        high_happiness["Quality of Life Index"].mean(),
        high_happiness["Purchasing Power Index"].mean(),
        high_happiness["Safety Index"].mean(),
        high_happiness["Pollution Index"].mean()
    ],

    "Low Happiness": [
        low_happiness["Quality of Life Index"].mean(),
        low_happiness["Purchasing Power Index"].mean(),
        low_happiness["Safety Index"].mean(),
        low_happiness["Pollution Index"].mean()
    ]
},

index=[
    "Quality of Life",
    "Purchasing Power",
    "Safety",
    "Pollution"
])

comparison = comparison.round(2)
display(comparison)

comparison.plot(
    kind="bar",
    figsize=(9,7)
)

plt.title("High vs Low Happiness Countries")
plt.xlabel("Indicator")
plt.ylabel("Mean Value")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)

plt.show()

Countries with higher happiness levels tend to have a higher Quality of Life Index, greater Purchasing Power and lower Pollution levels. Safety scores differ less between the two groups.

In [ ]:
import seaborn as sns

correlation = final_data[["Happiness Score", "Family", "Freedom", "Generosity", "Quality of Life Index", "Purchasing Power Index", "Safety Index", "Health Care Index", "Cost of Living Index", "Property Price to Income Ratio", "Traffic Commute Time Index", "Pollution Index"]].corr()

plt.figure(figsize=(9,7))

sns.heatmap(
    correlation,
    annot=True,
    cmap="Blues",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.show()

The heatmap reveals several strong relationships. Happiness Score is positively correlated with Quality of Life, Purchasing Power and Cost of Living indices, while showing a strong negative correlation with Pollution. Countries with better living conditions generally report higher levels of happiness.

### Hypothesis check


### Hypothesis 1:

The correlation between Purchasing Power Index and Happiness Score is stronger in countries with higher levels of Freedom.

In [ ]:
import numpy as np

final_data["Freedom Level"] = pd.cut(
    final_data["Freedom"],
    bins=3,
    labels=["Low Freedom", "Medium Freedom", "High Freedom"]
)

levels = ["Low Freedom", "Medium Freedom", "High Freedom"]
colors = ["#87CEFA", "#FF69B4", "#98FB98"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, level, color in zip(axes, levels, colors):
    subset = final_data[final_data["Freedom Level"] == level]
    r = subset["Purchasing Power Index"].corr(subset["Happiness Score"])

    ax.scatter(subset["Purchasing Power Index"], subset["Happiness Score"],
               alpha=0.6, color=color, s=40)

    m, b = np.polyfit(subset["Purchasing Power Index"], subset["Happiness Score"], 1)
    x_sorted = sorted(subset["Purchasing Power Index"])
    ax.plot(x_sorted, [m*x + b for x in x_sorted], color="black", linewidth=1.5)

    ax.set_title(f"{level}\nr = {r:.2f}", fontsize=11)
    ax.set_xlabel("Purchasing Power Index")
    ax.set_ylabel("Happiness Score")
    ax.set_xlim(final_data["Purchasing Power Index"].min() - 5,
                final_data["Purchasing Power Index"].max() + 5)
    ax.set_ylim(final_data["Happiness Score"].min() - 0.2,
                final_data["Happiness Score"].max() + 0.2)
    ax.grid(alpha=0.3)

plt.suptitle("Hypothesis 1: The correlation between Purchasing Power and Happiness\nis stronger in countries with higher Freedom",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
print("Correlation between Purchasing Power Index and Happiness Score by Freedom Level:")
for level in levels:
    subset = final_data[final_data["Freedom Level"] == level]
    r = subset["Purchasing Power Index"].corr(subset["Happiness Score"])
    print(f"  {level}: r = {r:.3f}  (n = {len(subset)})")

The results support the hypothesis. The correlation increases from 0.56 in the Low Freedom group to 0.75 in the High Freedom group, indicating a stronger relationship between Purchasing Power and Happiness in countries with higher levels of Freedom.

### Hypothesis 2:

The relationship between Purchasing Power Index and Happiness Score differs across countries with different Safety levels.

In [ ]:
final_data["Safety Level"] = pd.cut(
    final_data["Safety Index"],
    bins=3,
    labels=["Low Safety", "Medium Safety", "High Safety"]
)

early = final_data[final_data["Year"] <= 2018]
late = final_data[final_data["Year"] > 2018]

periods = [("2015–2018", early), ("2019–2022", late)]
levels = ["Low Safety", "Medium Safety", "High Safety"]
colors = ["#FFB347", "#C39BD3", "#76D7C4"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for row, (period_label, period_data) in enumerate(periods):
    for col, (level, color) in enumerate(zip(levels, colors)):
        ax = axes[row][col]
        subset = period_data[period_data["Safety Level"] == level]

        ax.scatter(subset["Purchasing Power Index"], subset["Happiness Score"],
                   alpha=0.6, color=color, s=40)

        if len(subset) > 2:
            m, b = np.polyfit(subset["Purchasing Power Index"], subset["Happiness Score"], 1)
            x_sorted = sorted(subset["Purchasing Power Index"])
            ax.plot(x_sorted, [m*x + b for x in x_sorted], color="black", linewidth=1.5)
            r = subset["Purchasing Power Index"].corr(subset["Happiness Score"])
            ax.set_title(f"{level}\n{period_label}  |  r = {r:.2f}", fontsize=11)
        else:
            ax.set_title(f"{level}\n{period_label}  |  n too small", fontsize=11)

        ax.set_xlabel("Purchasing Power Index")
        ax.set_ylabel("Happiness Score")
        ax.set_xlim(final_data["Purchasing Power Index"].min() - 5,
                    final_data["Purchasing Power Index"].max() + 5)
        ax.set_ylim(final_data["Happiness Score"].min() - 0.2,
                    final_data["Happiness Score"].max() + 0.2)
        ax.grid(alpha=0.3)

plt.suptitle(
    "Hypothesis 2: The relationship between Purchasing Power\n"
    "and Happiness differs across Safety Levels",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
print("\n2015–2018:")
for level in levels:
    s = early[early["Safety Level"] == level]
    r = s["Purchasing Power Index"].corr(s["Happiness Score"])
    print(f"  {level}: r = {r:.3f}, mean Happiness = {s['Happiness Score'].mean():.3f}, n = {len(s)}")

print("\n2019–2022:")
for level in levels:
    s = late[late["Safety Level"] == level]
    r = s["Purchasing Power Index"].corr(s["Happiness Score"])
    print(f"  {level}: r = {r:.3f}, mean Happiness = {s['Happiness Score'].mean():.3f}, n = {len(s)}")


The hypothesis is supported.

Countries with different Safety levels show different relationships between Purchasing Power Index and Happiness Score. Low Safety countries exhibit a weak or negative correlation, whereas Medium and High Safety countries show strong positive correlations. This suggests that Safety level may influence how strongly Purchasing Power is associated with Happiness.


### Hypothesis 3:

The three factors most strongly correlated with the Quality of Life Index are also the three factors most strongly correlated with the Happiness Score.

In [ ]:
cols = ["Family", "Freedom", "Generosity",
        "Purchasing Power Index", "Safety Index",
        "Health Care Index", "Cost of Living Index",
        "Property Price to Income Ratio",
        "Traffic Commute Time Index", "Pollution Index"]

corr_qol = (final_data[cols + ["Quality of Life Index"]]
            .corr()["Quality of Life Index"]
            .drop("Quality of Life Index")
            .abs()
            .sort_values(ascending=False))

corr_hap = (final_data[cols + ["Happiness Score"]]
            .corr()["Happiness Score"]
            .drop("Happiness Score")
            .abs()
            .sort_values(ascending=False))

top3_qol = list(corr_qol.head(3).index)
top3_hap = list(corr_hap.head(3).index)
overlap = set(top3_qol) & set(top3_hap)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

bars0 = axes[0].bar(corr_qol.head(5).index, corr_qol.head(5).values,
                     color=["#E74C3C" if f in overlap else "#F1948A" for f in corr_qol.head(5).index])
axes[0].set_title("Top-5 Factors Correlated\nwith Quality of Life Index", fontsize=11)
axes[0].set_ylabel("Absolute Correlation")
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_ylim(0, 1)
axes[0].grid(axis="y", alpha=0.3)

for bar, val in zip(bars0, corr_qol.head(5).values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01,
                 f"{val:.2f}", ha="center", va="bottom", fontsize=9)

bars1 = axes[1].bar(corr_hap.head(5).index, corr_hap.head(5).values,
                     color=["#8E44AD" if f in overlap else "#C39BD3" for f in corr_hap.head(5).index])
axes[1].set_title("Top-5 Factors Correlated\nwith Happiness Score", fontsize=11)
axes[1].set_ylabel("Absolute Correlation")
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_ylim(0, 1)
axes[1].grid(axis="y", alpha=0.3)

for bar, val in zip(bars1, corr_hap.head(5).values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.01,
                 f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Hypothesis 3: Do the same factors drive Quality of Life and Happiness?\n(darker bars = factor appears in top-3 of both indices)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
print("Top-3 correlated with Quality of Life Index:", top3_qol)
print("Top-3 correlated with Happiness Score:", top3_hap)
print("Overlap:", overlap)

The hypothesis is supported. The three factors with the strongest absolute correlations with the Quality of Life Index — Pollution Index (|r| = 0.84), Purchasing Power Index (|r| = 0.77), and Cost of Living Index (|r| = 0.68) — also appear among the top three factors by absolute correlation with Happiness Score. This gives a full overlap between the two top-3 lists.

### Data transformation

In [ ]:
final_data["Affordability"] = ( final_data["Purchasing Power Index"] / final_data["Cost of Living Index"])

final_data[["Country", "Year", "Affordability"]].head(10)

A new variable called Affordability was created by dividing Purchasing Power Index by Cost of Living Index. This indicator reflects the relationship between purchasing power and living expenses. Higher values indicate that residents have greater purchasing power relative to the cost of living, making everyday life more affordable.

In [ ]:
final_data["Health_and_Safety"] = (final_data["Health Care Index"] + final_data["Safety Index"]) / 2

final_data[["Country", "Year", "Health_and_Safety"]].head()

A new variable called Health_and_Safety was created as the average of Health Care Index and Safety Index. It combines healthcare quality and public safety into a single indicator, providing a broader measure of living conditions.

### Conclusion

This project investigated the relationship between socioeconomic factors, Quality of Life, and Happiness Score across countries from 2015 to 2022 by combining data from the World Happiness Report and the Quality of Life Index datasets. The merged dataset enabled a broader analysis of how economic, social, and environmental indicators are associated with national well-being.

The results showed that the relationship between Purchasing Power Index and Happiness Score is influenced by other social factors. Countries with higher levels of Freedom generally demonstrated a stronger correlation between purchasing power and happiness, suggesting that economic resources may be more closely connected to well-being in societies with greater personal freedom. In addition, the relationship between purchasing power and happiness differed across countries with different Safety levels, indicating that social conditions can affect how strongly economic factors are associated with happiness. The analysis also supported the third hypothesis. The three factors most strongly correlated with the Quality of Life Index were also among the strongest factors correlated with Happiness Score. This finding suggests that objective quality-of-life measures and subjective well-being are closely related and are influenced by similar socioeconomic conditions.

Overall, the project shows that national well-being is related to several economic and social factors at the same time. The merged dataset helped compare objective quality-of-life indicators with subjective happiness measures and showed that these two aspects of well-being are closely connected.